# B4 · GeoPandas ⇄ PostGIS — read_postgis and to_postgis

**What this is:** the bridge you will use most. Two commands:
- `gpd.read_postgis(sql, engine)` → runs SQL in PostGIS and returns a **GeoDataFrame**.
- `gdf.to_postgis(name, engine)` → writes a GeoDataFrame as a **table** in PostGIS.

Underneath: SQLAlchemy (B3) manages the connection, Psycopg (B2) talks to the server, Shapely unpacks the WKB geometries (B2 3.4).

**Daily picture:** PostGIS is a big warehouse, GeoPandas is your workbench. `read_postgis` is ordering exactly the parts you need
(ideally already cut to size in the warehouse). `to_postgis` is sending your finished product back to the warehouse for others to use.

**Key professional skill:** decide **where** each step happens. Heavy filtering and joins on big tables → SQL.
Exploring, modelling, plotting → GeoPandas.

⚙️ The database from **B0** must be running.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sqlalchemy import text
from geotrain.db import engine, sql
eng = engine()

---
## Level 1 — Basics: read and write whole tables

**Goal of this level:** load a PostGIS table into a GeoDataFrame and write one back.

**Daily-life picture:** Taking a box off the warehouse shelf, and putting a new box on it.

### Exercise 1.1 — Read a table

**Step 1 · Purpose.** What is the **purpose** of `gpd.read_postgis(sql, engine, geom_col='geometry')`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`sql="SELECT * FROM neighbourhoods"`, `con=eng`, `geom_col="geometry"`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
nbh = gpd.____("SELECT * FROM neighbourhoods", eng, geom_col="____")
print(nbh.crs, len(nbh))
nbh.plot(column="population", legend=True); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Runs a SQL query and returns a GeoDataFrame; `geom_col` names the geometry column. The CRS is read from the SRID.

**Real life:** Start an analysis from the city's central database instead of from files on your laptop.

```python
nbh = gpd.read_postgis("SELECT * FROM neighbourhoods", eng, geom_col="geometry")
print(nbh.crs, len(nbh))
nbh.plot(column="population", legend=True); plt.show()
```

</details>

### Exercise 1.2 — Write a new table

**Step 1 · Purpose.** What is the **purpose** of `gdf.to_postgis(name, engine, if_exists=...)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Build shops from the CSV as in A3 1.4, `to_crs(32633)`, then `to_postgis("shops", eng, if_exists="replace", index=False)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
df = pd.read_csv(DATA_DIR / "shops_wgs84.csv")
shops = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(32633)
shops.____("shops", eng, if_exists="____", index=False)
sql("SELECT kind, count(*) FROM shops GROUP BY kind")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Writes the GeoDataFrame to a PostGIS table; `if_exists` decides what happens if it exists: 'fail' (default), 'replace', or 'append'.

**Real life:** Publishing the shops layer (built from a CSV in A3) so a colleague's web map can use it.

```python
df = pd.read_csv(DATA_DIR / "shops_wgs84.csv")
shops = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(32633)
shops.to_postgis("shops", eng, if_exists="replace", index=False)
sql("SELECT kind, count(*) FROM shops GROUP BY kind")
```

</details>

### Exercise 1.3 — Check what arrived

**Step 1 · Purpose.** What is the **purpose** of checking the table after `to_postgis`?

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`SELECT type, srid FROM geometry_columns WHERE f_table_name = 'shops'`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sql("SELECT type, srid FROM geometry_columns WHERE f_table_name = '____'")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** To verify that the geometry type and SRID were stored as expected (a wrong or missing SRID breaks every later spatial query).

**Real life:** Quality control before telling colleagues 'the layer is online'.

```python
sql("SELECT type, srid FROM geometry_columns WHERE f_table_name = 'shops'")
```

</details>

---
## Level 2 — Core tools: let the database do the cutting

**Goal of this level:** filter and compute in SQL, then read only the result.

**Daily-life picture:** Asking the butcher to cut the meat for you instead of carrying home the whole animal.

### Exercise 2.1 — Filter in SQL with parameters

**Step 1 · Purpose.** What is the **purpose** of `read_postgis(text('... :p'), eng, params={...})`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Wrap the query in `text()` and use `:sev`; pass `params={"sev": "serious"}`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
serious = gpd.read_postgis(text("SELECT * FROM accidents WHERE severity = ____"), eng, geom_col="geometry", params={"sev": "____"})
print(len(serious))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Reads only the rows you need, with a safe parameter; less data travels over the network.

**Real life:** Load only the accidents of one severity (or one year) from a table of millions.

```python
serious = gpd.read_postgis(text("SELECT * FROM accidents WHERE severity = :sev"), eng, geom_col="geometry", params={"sev": "serious"})
print(len(serious))
```

</details>

### Exercise 2.2 — Compute geometry in SQL, receive the result

**Step 1 · Purpose.** What is the **purpose** of `SELECT ST_Intersection(...) AS geom ...`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Reuse B1 2.1 but return the geometry: `ST_Intersection(n.geometry, ST_Buffer(r.geometry, 150)) AS geom`, `geom_col="geom"`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
q = """
    SELECT n.name, ST_Intersection(n.geometry, ST_Buffer(r.geometry, 150)) AS geom
    FROM neighbourhoods n JOIN river r ON ST_Intersects(n.geometry, ST_Buffer(r.geometry, 150))
"""
flood_parts = gpd.read_postgis(q, eng, geom_col="____")
ax = nbh.boundary.plot(color="grey"); flood_parts.plot(ax=ax, column="name", legend=True); plt.show()
flood_parts.assign(ha=flood_parts.area / 1e4)[["name", "ha"]].round(1)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** PostGIS builds new geometries (here flooded parts of neighbourhoods) and GeoPandas receives them ready to map.

**Real life:** A server computes the heavy overlay once; many users just download the small result.

```python
q = """
    SELECT n.name, ST_Intersection(n.geometry, ST_Buffer(r.geometry, 150)) AS geom
    FROM neighbourhoods n JOIN river r ON ST_Intersects(n.geometry, ST_Buffer(r.geometry, 150))
"""
flood_parts = gpd.read_postgis(q, eng, geom_col="geom")
ax = nbh.boundary.plot(color="grey"); flood_parts.plot(ax=ax, column="name", legend=True); plt.show()
flood_parts.assign(ha=flood_parts.area / 1e4)[["name", "ha"]].round(1)
```

</details>

### Exercise 2.3 — Add rows to a table

**Step 1 · Purpose.** What is the **purpose** of `to_postgis(..., if_exists='append')`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Make 2 new shops as a GeoDataFrame with the same columns, then append and count again.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
new = shops.head(2).copy()
new["shop_id"] = [41, 42]
new.to_postgis("shops", eng, if_exists="____", index=False)
sql("SELECT count(*) FROM shops")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Adds the new rows at the end of an existing table (columns must match).

**Real life:** Monthly accident reports added to the same table (the pipeline from A4's test, now into PostGIS).

```python
new = shops.head(2).copy()
new["shop_id"] = [41, 42]
new.to_postgis("shops", eng, if_exists="append", index=False)
sql("SELECT count(*) FROM shops")
```

</details>

### Exercise 2.4 — Read in chunks

**Step 1 · Purpose.** What is the **purpose** of `read_postgis(..., chunksize=n)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Loop: `for part in gpd.read_postgis(..., chunksize=500):` and add up `part["residents"].sum()`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
total = 0
for part in gpd.read_postgis("SELECT * FROM houses", eng, geom_col="geometry", chunksize=____):
    total += part["residents"].sum()
total

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Returns an iterator of GeoDataFrames of n rows each, instead of one huge GeoDataFrame.

**Real life:** Processing a national buildings table piece by piece on a normal laptop.

```python
total = 0
for part in gpd.read_postgis("SELECT * FROM houses", eng, geom_col="geometry", chunksize=500):
    total += part["residents"].sum()
total
```

</details>

---
## Level 3 — Combining: round trips between the two worlds

**Goal of this level:** compute in GeoPandas, store in PostGIS, query again in SQL — and use views.

**Daily-life picture:** A relay race: each runner (tool) runs the part of the track they are best at.

### Exercise 3.1 — GeoPandas result → PostGIS → SQL question

**Step 1 · Purpose.** What is the **purpose** of storing an analysis result in PostGIS instead of a file?

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Compute the rate with `gpd.sjoin` as in A3, write `nbh_rates` to PostGIS, then query the top 3 in SQL.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
acc = gpd.read_postgis("SELECT * FROM accidents", eng, geom_col="geometry")
counts = gpd.sjoin(acc, nbh[["name", "geometry"]], predicate="within")["name"].value_counts()
nbh["acc_per_1000"] = nbh["name"].map(counts).fillna(0) / nbh["population"] * 1000
nbh[["name", "population", "acc_per_1000", "geometry"]].____("nbh_rates", eng, if_exists="replace", index=False)
sql("SELECT name, round(acc_per_1000::numeric, 2) AS rate FROM nbh_rates ORDER BY rate DESC LIMIT 3")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** The result becomes a shared, queryable table: other people, apps or SQL queries can use it immediately.

**Real life:** Your 'accidents per 1000 residents' layer feeds the city's dashboard every morning.

```python
acc = gpd.read_postgis("SELECT * FROM accidents", eng, geom_col="geometry")
counts = gpd.sjoin(acc, nbh[["name", "geometry"]], predicate="within")["name"].value_counts()
nbh["acc_per_1000"] = nbh["name"].map(counts).fillna(0) / nbh["population"] * 1000
nbh[["name", "population", "acc_per_1000", "geometry"]].to_postgis("nbh_rates", eng, if_exists="replace", index=False)
sql("SELECT name, round(acc_per_1000::numeric, 2) AS rate FROM nbh_rates ORDER BY rate DESC LIMIT 3")
```

</details>

### Exercise 3.2 — A view: a saved query that behaves like a table

**Step 1 · Purpose.** What is the **purpose** of `CREATE VIEW ... AS SELECT ...`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Create the view with `sql(...)` (the helper commits). Then `read_postgis("SELECT * FROM v_serious", ...)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sql("""
    CREATE OR REPLACE ____ v_serious AS
    SELECT n.name, count(a.*) AS serious, n.geometry
    FROM neighbourhoods n LEFT JOIN accidents a
         ON ST_Within(a.geometry, n.geometry) AND a.severity IN ('serious', 'fatal')
    GROUP BY n.name, n.geometry
""")
v = gpd.read_postgis("SELECT * FROM v_serious", eng, geom_col="geometry")
v.plot(column="serious", legend=True, cmap="Reds"); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Stores a query under a name; reading the view runs the query on the latest data.

**Real life:** 'serious_accidents_by_neighbourhood' always up to date, without anybody re-running a script.

```python
sql("""
    CREATE OR REPLACE VIEW v_serious AS
    SELECT n.name, count(a.*) AS serious, n.geometry
    FROM neighbourhoods n LEFT JOIN accidents a
         ON ST_Within(a.geometry, n.geometry) AND a.severity IN ('serious', 'fatal')
    GROUP BY n.name, n.geometry
""")
v = gpd.read_postgis("SELECT * FROM v_serious", eng, geom_col="geometry")
v.plot(column="serious", legend=True, cmap="Reds"); plt.show()
```

</details>

---
## Level 4 — Professional: hybrid workflows

**Goal of this level:** split each question between SQL and GeoPandas on purpose, and explain the split.

**Daily-life picture:** A kitchen team: the prep cook cuts (SQL), the chef finishes and plates (GeoPandas).

### Exercise 4.1 — School demand model — heavy part in SQL, model in GeoPandas

**Scenario.** Repeat A3 4.4 (nearest primary school, 5 % children) but let **PostGIS** assign every house to its nearest primary school with a KNN `LATERAL` join, and return only the **summed demand per school** with its geometry. Then compute the load and map it in GeoPandas.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

SQL: `FROM houses h CROSS JOIN LATERAL (SELECT school FROM schools WHERE level='primary' ORDER BY geometry <-> h.geometry LIMIT 1) s`, `GROUP BY s.school`. Join back to schools for capacity and geometry.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
q = """
    WITH alloc AS (
        SELECT s.school, sum(h.residents) * 0.05 AS demand
        FROM houses h
        CROSS JOIN LATERAL (SELECT school FROM schools WHERE level = 'primary'
                            ORDER BY geometry ____ h.geometry LIMIT 1) s
        GROUP BY s.school)
    SELECT sc.school, sc.capacity, a.demand, sc.geometry
    FROM schools sc JOIN alloc a USING (school)
"""
load = gpd.read_postgis(q, eng, geom_col="geometry")
load["load"] = (load["demand"] / load["capacity"]).round(2)
ax = nbh.boundary.plot(color="grey"); load.plot(ax=ax, column="load", cmap="RdYlGn_r", legend=True, markersize=200); plt.show()
load.drop(columns="geometry")

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Modelling (hybrid workflow)

Same numbers as A3 4.4, but only **4 rows** travel from the database instead of 1 604 houses. With 20 million houses, this split is the difference between seconds and hours.

```python
q = """
    WITH alloc AS (
        SELECT s.school, sum(h.residents) * 0.05 AS demand
        FROM houses h
        CROSS JOIN LATERAL (SELECT school FROM schools WHERE level = 'primary'
                            ORDER BY geometry <-> h.geometry LIMIT 1) s
        GROUP BY s.school)
    SELECT sc.school, sc.capacity, a.demand, sc.geometry
    FROM schools sc JOIN alloc a USING (school)
"""
load = gpd.read_postgis(q, eng, geom_col="geometry")
load["load"] = (load["demand"] / load["capacity"]).round(2)
ax = nbh.boundary.plot(color="grey"); load.plot(ax=ax, column="load", cmap="RdYlGn_r", legend=True, markersize=200); plt.show()
load.drop(columns="geometry")
```

</details>

### Exercise 4.2 — Monte Carlo test with random points made in PostGIS

**Scenario.** Repeat A3 4.3: are accidents closer to primary roads than random points? Generate the random points **in PostGIS** with `ST_GeneratePoints` (99 sets of 260 points inside Riverton) and read back only the mean distance per set.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`generate_series(1, 99) AS sim` × `ST_Dump(ST_GeneratePoints(town, 260, sim))` (the third argument is a seed, so results repeat). Mean distance to the union of primary roads, grouped by `sim`.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
q = """
    WITH town AS (SELECT ST_Union(geometry) AS g FROM neighbourhoods),
         prim AS (SELECT ST_Union(geometry) AS g FROM roads WHERE road_type = 'primary'),
         pts AS (SELECT sim, (ST_Dump(____(town.g, 260, sim))).geom AS g
                 FROM town, generate_series(1, 99) AS sim)
    SELECT sim, avg(ST_Distance(pts.g, prim.g)) AS mean_d FROM pts, prim GROUP BY sim
"""
sims = sql(q)["mean_d"].astype(float)
real = float(sql("SELECT avg(ST_Distance(a.geometry, p.g)) FROM accidents a, (SELECT ST_Union(geometry) g FROM roads WHERE road_type='primary') p").iloc[0, 0])
p = ((sims <= real).sum() + 1) / (len(sims) + 1)
print(f"real {real:.0f} m, random {sims.mean():.0f} m, p = {p:.2f}")

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Statistical (simulation in SQL)

Same conclusion as in GeoPandas (p = 0.01, the smallest possible with 99 runs). The heavy simulation ran on the server; Python only did the final comparison.

```python
q = """
    WITH town AS (SELECT ST_Union(geometry) AS g FROM neighbourhoods),
         prim AS (SELECT ST_Union(geometry) AS g FROM roads WHERE road_type = 'primary'),
         pts AS (SELECT sim, (ST_Dump(ST_GeneratePoints(town.g, 260, sim))).geom AS g
                 FROM town, generate_series(1, 99) AS sim)
    SELECT sim, avg(ST_Distance(pts.g, prim.g)) AS mean_d FROM pts, prim GROUP BY sim
"""
sims = sql(q)["mean_d"].astype(float)
real = float(sql("SELECT avg(ST_Distance(a.geometry, p.g)) FROM accidents a, (SELECT ST_Union(geometry) g FROM roads WHERE road_type='primary') p").iloc[0, 0])
p = ((sims <= real).sum() + 1) / (len(sims) + 1)
print(f"real {real:.0f} m, random {sims.mean():.0f} m, p = {p:.2f}")
```

</details>

### Exercise 4.3 — Accidents per month and severity

**Scenario.** Count accidents per month and severity with SQL (`date_trunc`), read the small table, and draw a stacked bar chart.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`SELECT date_trunc('month', date) AS month, severity, count(*) ... GROUP BY 1, 2`, then `pivot` in pandas and `.plot(kind='bar', stacked=True)`.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
t = sql("SELECT ____('month', date)::date AS month, severity, count(*) AS n FROM accidents GROUP BY 1, 2 ORDER BY 1")
t.pivot(index="month", columns="severity", values="n").fillna(0).plot(kind="bar", stacked=True, figsize=(8, 3)); plt.show()

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Temporal (aggregate in SQL, plot in pandas)

Non-spatial aggregation is also best done where the data lives. Only 36 numbers come back instead of 260 rows (or 2.6 million in a real city).

```python
t = sql("SELECT date_trunc('month', date)::date AS month, severity, count(*) AS n FROM accidents GROUP BY 1, 2 ORDER BY 1")
t.pivot(index="month", columns="severity", values="n").fillna(0).plot(kind="bar", stacked=True, figsize=(8, 3)); plt.show()
```

</details>

---
## 🏁 Final test

For each: **question type → where does each step happen (SQL or GeoPandas) and why → code → interpretation.**

### Test 1 — 🛠️ Task question

**A.** Read the houses within **400 m of Station Avenue** as a GeoDataFrame (filter in SQL), and plot them over the neighbourhoods.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
near = gpd.read_postgis(text("""
    SELECT h.* FROM houses h JOIN roads r ON ST_DWithin(h.geometry, r.geometry, :d)
    WHERE r.name = :road"""), eng, geom_col="geometry", params={"d": 400, "road": "Station Avenue"})
ax = nbh.boundary.plot(color="grey"); near.plot(ax=ax, markersize=3); plt.show()
print(len(near))
```
Filter in SQL (index, fewer rows); plot in GeoPandas.

</details>

### Test 2 — 🛠️ Task question

**B.** Compute, per neighbourhood, the **number of shops of each kind** in SQL and save the wide table (one column per kind) with geometry to PostGIS as `nbh_shops`.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
q = """SELECT n.name, n.geometry,
            count(*) FILTER (WHERE s.kind = 'bakery')   AS bakery,
            count(*) FILTER (WHERE s.kind = 'grocery')  AS grocery,
            count(*) FILTER (WHERE s.kind = 'pharmacy') AS pharmacy,
            count(*) FILTER (WHERE s.kind = 'cafe')     AS cafe
       FROM neighbourhoods n LEFT JOIN shops s ON ST_Within(s.geometry, n.geometry)
       GROUP BY n.name, n.geometry"""
wide = gpd.read_postgis(q, eng, geom_col="geometry")
wide.to_postgis("nbh_shops", eng, if_exists="replace", index=False)
wide.drop(columns="geometry")
```

</details>

### Test 3 — 🛠️ Task question

**C.** Your GeoDataFrame of candidate clinic sites (A3 4.5) must go to the planning department's database, with the score, in EPSG:4326. Write the code.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
cand = gpd.GeoDataFrame({"score": [997.0, 850.0]}, geometry=gpd.points_from_xy([394_125, 391_625], [5_820_625, 5_822_125]), crs=32633)
cand.to_crs(4326).to_postgis("clinic_candidates", eng, if_exists="replace", index=False)
sql("SELECT srid, type FROM geometry_columns WHERE f_table_name = 'clinic_candidates'")
```

</details>

### Test 4 — 🧠 Modelling question

**D · Modelling the workflow.** Riverton wants a **weekly accessibility dashboard**: for each neighbourhood, % of residents within 800 m of a clinic, a school and a pharmacy.
Design the workflow: which steps in PostGIS, which in GeoPandas, what is stored, and how is it refreshed?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

1. **PostGIS** holds the source tables (houses, clinics, schools, shops) — updated by loading jobs (Psycopg COPY / `to_postgis(append)`).
2. **PostGIS view or materialized view** `access_by_nbh`: per house `EXISTS(... ST_DWithin ...)` for each service, aggregated per neighbourhood (B1 4.2). `REFRESH MATERIALIZED VIEW` weekly (a scheduled job).
3. **GeoPandas** `read_postgis("SELECT * FROM access_by_nbh")` → maps and charts for the report / dashboard.
Why: heavy spatial work near the data, with indexes; only 9 rows travel to Python; everyone sees the same numbers.

</details>

---
## 🏗️ Project — Bike share: which stations work, and where is the gap?

**The brief.** Combine SQL and GeoPandas on purpose. The trip counting (6 000 trips) happens **in PostGIS**; the catchment analysis
        (residents around each station) happens **in GeoPandas**; the results go back to PostGIS for QGIS. Finally, find the tract where a
        new station would reach the most people who have none today.

**Your data** (made for this project, in `data/projects/`):

| Layer / file | What it holds |
|---|---|
| `bike_stations.csv, bike_trips.csv` | stations (lon/lat) and trips |
| `projects.buildings, projects.tracts` | residents and tracts (B0 project) |

**How to work:** for each task write the plan in words, then the code, then one sentence with the result. Model answers are hidden; open them only after you tried. The last part happens in **QGIS**.

In [ ]:
from geotrain.projects import build_projects, PROJ_DIR, publish
build_projects()

import numpy as np

### Project task 1

**Load** the stations as points in EPSG:32633 into `projects.bike_stations` (GeoPandas) and the trips as a plain table `projects.bike_trips`.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
st = pd.read_csv(PROJ_DIR / "bike_stations.csv")
st = gpd.GeoDataFrame(st, geometry=gpd.points_from_xy(st.lon, st.lat), crs=4326).to_crs(32633)
st.to_postgis("bike_stations", eng, schema="projects", if_exists="replace", index=False)
pd.read_csv(PROJ_DIR / "bike_trips.csv").to_sql("bike_trips", eng, schema="projects", if_exists="replace", index=False)
sql("SELECT (SELECT count(*) FROM projects.bike_stations) st, (SELECT count(*) FROM projects.bike_trips) trips")
```

</details>

### Project task 2

**Count in the database:** starts and ends per station, read back **with geometry** (`read_postgis`).

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
use = gpd.read_postgis("""
    SELECT s.station_id, s.name, s.capacity, s.geometry,
           (SELECT count(*) FROM projects.bike_trips t WHERE t.start_station = s.station_id) AS starts,
           (SELECT count(*) FROM projects.bike_trips t WHERE t.end_station = s.station_id) AS ends
    FROM projects.bike_stations s""", eng, geom_col="geometry")
use.nlargest(3, "starts")[["name", "starts", "ends"]]
```

</details>

### Project task 3

**Catchments in GeoPandas:** residents within 400 m of each station (building centroids), and trips per 100 residents. Which station performs best relative to its catchment?

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
bld = gpd.read_postgis("SELECT residents, ST_Centroid(geometry) AS geometry FROM projects.buildings WHERE residents > 0", eng, geom_col="geometry")
ring = use[["station_id", "geometry"]].assign(geometry=use.buffer(400))
res = gpd.sjoin(bld, ring, predicate="within").groupby("station_id")["residents"].sum()
use["residents_400m"] = use["station_id"].map(res).fillna(0)
use["starts_per_100res"] = (use["starts"] / use["residents_400m"].replace(0, np.nan) * 100).round(1)
use.sort_values("starts_per_100res", ascending=False)[["name", "starts", "residents_400m", "starts_per_100res"]].head()
```

</details>

### Project task 4

**The gap (in SQL):** per tract, residents living **more than 500 m** from any station (`NOT EXISTS … ST_DWithin`). Which tract is the best candidate for a new station?

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
gap = gpd.read_postgis("""
    SELECT t.tract_id, t.geometry, coalesce(sum(b.residents), 0) AS unserved
    FROM projects.tracts t
    LEFT JOIN projects.buildings b ON ST_Within(ST_Centroid(b.geometry), t.geometry) AND b.residents > 0
         AND NOT EXISTS (SELECT 1 FROM projects.bike_stations s WHERE ST_DWithin(s.geometry, b.geometry, 500))
    GROUP BY t.tract_id, t.geometry""", eng, geom_col="geometry")
gap.nlargest(3, "unserved")[["tract_id", "unserved"]]
```

</details>

### Project task 5

**Publish** `station_performance` (task 3) and `station_gap` (task 4) to PostGIS.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
use.to_postgis("station_performance", eng, schema="projects", if_exists="replace", index=False)
gap.to_postgis("station_gap", eng, schema="projects", if_exists="replace", index=False)
sql("SELECT f_table_name FROM geometry_columns WHERE f_table_schema = 'projects' AND f_table_name LIKE 'station%'")
```
**Why this split:** counting 6 000 (or 6 million) trips is a database job; buffers, joins and ratios for 24 stations are quick and flexible in GeoPandas.

</details>

### 🗺️ Project in QGIS

1. Open `riverton_projects.qgz`, add **station_performance** and **station_gap**.
2. `station_gap`: *Graduated* on `unserved`; `station_performance`: circles sized by `starts`, labelled with `starts_per_100res`.
3. With the *Add Point Feature* tool, digitise your proposed new station in a new *Temporary scratch layer*, and measure its distance to the nearest stations.
4. Build a print layout: title, the two layers, legend, and a text box with your recommendation.

<details><summary>🔌 <b>How to connect QGIS to PostGIS</b> (once per computer)</summary>

1. Start the database (Docker: `docker start geotrain-db`).
2. In QGIS, open the **Browser** panel → right-click **PostgreSQL** → **New Connection…**
3. Name `geotrain`, Host `localhost`, Port `5432`, Database `geotrain`.
   Authentication → *Basic*: user `geo`, password `geo` (tick *Store*). Click **Test Connection** → OK.
4. Expand **geotrain → projects** (the schema) and **drag a table onto the map**.
5. To run SQL and see the result on the map: **Database → DB Manager → PostGIS → geotrain → SQL Window**.
   Write the query → **Execute** → tick **Load as new layer**, choose the geometry column and a unique id column → **Load**.
6. To open a GeoPackage instead: drag the `.gpkg` file from your file manager into QGIS.

</details>

### 📦 Deliverables (tick when done)

- [ ] Station usage table (from SQL)
- [ ] Catchment and performance table (from GeoPandas)
- [ ] Gap map with the recommended tract + short justification

### 🪞 Reflection — your long-term plan

Draw (on paper) your own ideal workflow: where does your data live, where is it analysed, where is the result published? Which B-notebook skills does each arrow need?

✍️ **Your answer:** …